**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/haley-2018"], check=True)
    os.chdir("/content/encore/entries/haley-2018")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Haley, Hampton & Marder (2018) from OSF project r7aes

**Paper.** Haley JA, Hampton D, Marder E (2018). Two central pattern generators from the crab, *Cancer borealis*, respond robustly and differentially to extreme extracellular pH. eLife 7:e41877, doi:10.7554/elife.41877.

**Dataset.** OSF project r7aes, "Recordings of stomatogastric and cardiac ganglia of C. borealis at varying pH" (doi:10.17605/osf.io/r7aes), files in OSF Storage. An OSF project is not versioned as a whole; each file is read at the OSF Storage version the registry's target list pinned (`r7aes_files.py`, built when the project was last modified, 2019-01-08T14:49:50), and the SHA-256 of every file read whole is checked against it. The paper cites the project without a version; these are the current versions of every file (checked against the live OSF listing below), and no file has a later version, so the choice is the only one.

**Text read.** The claims file `claims/haley-2018.yaml` was written from the version of record (Europe PMC, PMC6328273) and the paper's code, github.com/jesshaley/haley_hampton_marder_2018 at commit 83f5904. Methods it does not fix are taken from the paper and cited where they are used, and in the helper module `crabph.py`.

**What the dataset holds.** One folder per experiment, each with one Axon .abf file per pH step. The STG and CG folders have a README naming each file's pH and each channel's nerve; the nine picrotoxin (PTX) folders have none. The manual state annotations (`stateData_STG.xlsx`, `stateData_CG.xlsx`), the Spike2 spike and burst output and the gastric annotations are not in the project.

| Claim | Tested here |
|---|---|
| fig2a-stg1-traces | tested; STG states read from the pdn and lvn (assumption) |
| fig2bcd-stg1-measures | tested |
| fig3a-stg-variability-examples | blocked: needs the states 1 and 2 apart, which the spikes read here cannot tell (the parts that can be read are shown) |
| fig3c-stg-rhythmic-fraction | tested; STG states read from the pdn and lvn |
| stg-state-transitions | blocked: needs the full manual state annotation |
| fig3d-stg-frequency, fig3e-stg-spikes-per-burst, fig3f-stg-duty-cycle | tested |
| fig3s2-stg-control-variability | tested |
| fig4a-ptx-example, fig4bc-membrane-potential, fig4de-spike-amplitude, fig4fg-ptx-frequency | tested |
| gastric-like-rhythm-counts, gastric-conditions-co-occur, fig5-gastric-recovery-example | blocked: gastric rhythms were identified by eye, with no criterion (the nerves each experiment records are listed) |
| fig6-cg1-measures, fig6a-cg2-less-rhythmic | tested; CG state 1 from LC bursting |
| fig7b-cg-rhythmic-fraction | tested; CG state 1 from LC bursting |
| cg-state-transitions | blocked: needs the full manual state annotation |
| fig7c-cg-frequency, fig7d-cg-spikes-per-burst, fig7e-cg-duty-cycle, fig7s2-cg-control-variability | tested |
| fig8a-ganglion-difference, fig8b-individual-preparations, fig8b-cross-predictive | tested; states read from the spikes |
| baseline-frequency-correlations, robust-6-1-to-8-8, intact-depolarize-in-acid | tested |

**Revision after the first verification round.** The STG fraction rhythmic now requires LP and PY firing on the lvn between PD bursts (state 3 no longer counts), with PD bursting alone shown as a variant. Added: an LC isolation check with height histograms per unit, a PD label check of the PTX files that allows for the conduction delay to the pdn, the electrode list of every STG experiment, the testable parts of fig3a, and the variants with missing steps left out for Figure 8. OSF is read through WaterButler with a throttle, as osf.io answered the first round's request rate with 429.

**Revision after the second verification round.** Added: the response of each PTX experiment's bath pH electrode (slope against the nominal step), the lags and chance level of the 903_046 label check, the CG measures and Figure 8A without three more units whose isolation is doubtful at one step, and which of the fig3a preparations lack a 10.4 file. Several conclusions were corrected to match the printed numbers.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource
import numpy as np
import pandas as pd
from scipy import stats
import crabph as C

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
T0 = time.time()
print("OSF project", C.OSF_NODE, "files pinned as of", C.PUBLISHED_VERSION, "-", len(C.FILES), "files")

## The files on OSF now, against the pinned list

The listing walks every storage provider of the project through the OSF API (docs/archives.md, "OSF").

In [ ]:
try:
    listing, providers = C.osf_listing()
    print("providers:", providers)
    now = listing[listing.provider == "osfstorage"].set_index("path")
    changed = [p for p, f in C.FILES.items() if p not in now.index or now.loc[p, "version"] != f["version"]
               or now.loc[p, "sha256"] != f["sha256"] or now.loc[p, "size"] != f["size"]]
    print(f"osfstorage files now: {len(now)}; pinned: {len(C.FILES)}; changed or gone since pinned: {len(changed)}; "
          f"new: {len(set(now.index) - set(C.FILES))}")
except Exception as e:     # the listing is a check of the pin, not an input to any result
    print("OSF listing could not be read:", repr(e), "- the pin was not checked against OSF in this run")

## The experiments: READMEs, files, conditions and channels

The claims file lists 15 STG experiments, 15 CG units and 9 PTX experiments, from the paper's code. For each STG and CG experiment the README gives the file of each pH step; the code's 13 conditions are formed from that order (`crabph.protocol_columns`). The PD unit is read on the pdn and a CG unit "CGnL" on the trunk channel the README calls "CGn (trunk)".

In [ ]:
plans, missing_exps, jobs = C.plan_extracellular()
print("experiments listed by the paper's code and absent from the project:", missing_exps)
rows = []
for e, pl in plans.items():
    rows.append(dict(exp=e, order=pl["order"], files_in_readme=len(pl["seq_raw"]), files_used=len(pl["seq"]),
                     not_in_readme=", ".join(f.split("/")[1] for f in pl["unlisted"]),
                     readme_names_not_in_folder=", ".join(f"{e}: {p}" for f, p in zip([x for x, _ in pl["desc"]["files"]], [y for _, y in pl["seq_raw"]]) if False),
                     missing_conditions=", ".join(c for c, v in pl["columns"].items() if v is None),
                     units=", ".join(f"{u}:{ch}" for u, (g, ch) in pl["units"].items()),
                     intracellular=", ".join(f"{u}:{ch}" for u, ch in pl["intra"].items()),
                     notes=pl["desc"]["notes"]))
exp_table = pd.DataFrame(rows).drop(columns="readme_names_not_in_folder")
print(exp_table.to_string(index=False))

## Read the data

Every STG and CG file is streamed whole once, its SHA-256 checked, and the channels kept that the analysis uses: the unit channels in the first 10 min (for the code's window variant) and the last 8 min, the intracellular channels in the last minute, and the bath pH signal. Spikes are found as each file is read, so only event times and heights are held. The PTX files are read only over their last minute (the [INTRA] window), so their hashes are not checked. The time, the volume read whole and the peak memory are printed below.

In [ ]:
t = time.time()
ext_results = C.run_jobs(jobs, workers=8)
print(f"STG and CG: {len(ext_results)} files in {(time.time() - t) / 60:.1f} min; "
      f"{sum(C.FILES[j['path']]['size'] for j in jobs if j['whole']) / 1e9:.1f} GB read whole")
t = time.time()
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(12) as ex:
    ptx_jobs = [j for js in ex.map(C.ptx_jobs, C.PTX_EXPERIMENTS) for j in js]
    ptx_results = list(ex.map(C.process_ptx, ptx_jobs))
print(f"PTX: {len(ptx_results)} files in {(time.time() - t) / 60:.1f} min; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
print("SHA-256 of the STG and CG files read whole:", sum(r["sha_ok"] for r in ext_results), "match of", len(ext_results),
      "; mismatches:", [r["path"] for r in ext_results if not r["sha_ok"]])
heads = {f: h for pl in plans.values() for f, h in pl["heads"].items()}
H = pd.DataFrame(heads.values())
print("sampling rates (Hz) by experiment:", H.groupby(H.path.str[:7]).rate.unique().map(list).to_dict())
print("file lengths: %.1f to %.1f min (median %.1f)" % (H.seconds.min() / 60, H.seconds.max() / 60, H.seconds.median() / 60))
print("files shorter than the 8 min window:", [(r["path"], round(r["seconds"] / 60, 1)) for r in ext_results if r["seconds"] < C.WINDOW_S])
used = sorted({ch for pl in plans.values() for _, ch in pl["units"].values() if ch} | {ch for pl in plans.values() for ch in pl["intra"].values()})
unit_of = {}
for h in heads.values():
    for c, u in zip(h["channels"], h["units"]):
        if c in used:
            unit_of.setdefault((c, u), 0)
            unit_of[(c, u)] += 1
print("units of the channels used (channel, unit): files:", unit_of)
print("STG experiments with a PD unit:", sum(1 for pl in plans.values() for u, (g, ch) in pl["units"].items() if g == "STG" and ch),
      "of", len(C.STG_EXPERIMENTS), "listed; CG units:", sum(1 for pl in plans.values() for u, (g, ch) in pl["units"].items() if g == "CG" and ch),
      "of", len(C.CG_UNITS), "listed; PTX experiments:", len({r['path'][:7] for r in ptx_results}))
# the bath pH signal against the README's pH, per experiment (Spearman over the files of the step protocols)
R = {r["path"]: r for r in ext_results}
rows = []
for e, pl in plans.items():
    pairs = [(float(p), R[f]["ph"]) for f, p in pl["seq"] if R[f]["ph"] is not None]
    if len(pairs) > 3:
        rho = stats.spearmanr([a for a, _ in pairs], [b for _, b in pairs]).statistic
        rows.append(dict(exp=e, n_files=len(pairs), spearman_readme_pH_vs_signal=round(rho, 3)))
    else:
        rows.append(dict(exp=e, n_files=len(pairs), spearman_readme_pH_vs_signal=None))
print(pd.DataFrame(rows).to_string(index=False))
print("All channels of a file share its sampling clock (one interleaved ABF data block), and every measure is taken within a file, "
      "so no stream has to be aligned with another.")

The bath pH electrode signal is uncalibrated (its sign and scale differ between rigs), so it is compared with the README by rank: a negative correlation means the signal falls as the README's pH rises, as it does for a pH electrode in these recordings. The PTX folders have no README; the conditions of their files are taken from that signal (`crabph.assign_ptx_conditions`).

In [ ]:
ptx_files, ptx = C.ptx_tables(ptx_results)
print(ptx_files.assign(est_pH=ptx_files.est_pH.round(2), ph_signal=ptx_files.ph_signal.round(1), minutes=ptx_files.minutes.round(1),
                       PD_pdn_match=ptx_files.PD_pdn_match.round(2), LP_pdn_match=ptx_files.LP_pdn_match.round(2)).to_string(index=False))

In [ ]:
# how each experiment's bath electrode responds: its signal against the nominal step of each file used
print("bath pH signal against the nominal step, per PTX experiment (files used; slope in mV per pH unit):")
for e, d in ptx_files[ptx_files.used].groupby("exp"):
    x = d.label.astype(float)
    fit = stats.linregress(x, d.ph_signal)
    print(f"  {e}: slope {fit.slope:.1f}, r = {fit.rvalue:.4f}; signal at 5.5 {d.ph_signal[x == 5.5].mean():.1f}, "
          f"at 7.8 {d.ph_signal[x == 7.8].mean():.1f}, at 10.4 {d.ph_signal[x == 10.4].mean():.1f} mV")

In [ ]:
folders = sorted({p.split("/")[0] for p in C.FILES})
listed = sorted(set(C.STG_EXPERIMENTS) | {e for e, _ in C.CG_UNITS} | set(C.PTX_EXPERIMENTS))
print("experiments in the paper's code:", len(listed), "; folders in the project:", len(folders))
print("in the code, not in the project:", sorted(set(listed) - set(folders)))
print("STG 877_127 is one of the 15 STG preparations; CG 887_017 (unit CG3L) one of the 15 CG units.")

In [ ]:
for e, pl in plans.items():
    miss = [c for c, v in pl["columns"].items() if v is None]
    if miss:
        print(e, "README lists", len(pl["seq_raw"]), "files; no file for pH", miss)
for e in C.PTX_EXPERIMENTS:
    f = ptx_files[ptx_files.exp == e]
    have = set(f.label)
    miss = [s for s in C.ACID_STEPS + C.BASE_STEPS if s not in have]
    if miss:
        print(e, "(PTX, from the recorded pH)", len(f), "files; no file for pH", miss)

In [ ]:
for e, pl in plans.items():
    names = [f for f, _ in pl["desc"]["files"]]
    absent = [(f, r) for f, (r, _) in zip(names, pl["seq_raw"]) if f"{e}/{f}" not in C.FILES]
    if absent:
        print(e, "README names", len(absent), "files not in the folder, e.g.", absent[0][0], "-> matched by number to", absent[0][1])
    if pl["unlisted"]:
        print(e, "folder holds files the README does not list:", [f.split("/")[1] for f in pl["unlisted"]])
    keys = [k for k, _ in pl["desc"]["electrodes"] if k.startswith(("IN", "extra"))]
    per_file = [set(h["channels"]) for h in pl["heads"].values()]
    never = [k for k in keys if not any(k in ch for ch in per_file)]
    some = [k for k in keys if k not in never and not all(k in ch for ch in per_file)]
    if never or some:
        print(e, "README electrodes in no file:", never, "; in only some files:", some)

In [ ]:
print("PTX folders have no README; pH per file from the recorded bath pH signal (mV):")
for e in ["903_058", "903_060"]:
    f = ptx_files[ptx_files.exp == e]
    print(e, [(r.file.split("_")[-1][:4], r.label, round(r.ph_signal, 1)) for r in f.itertuples()])
print("In both, the file recorded after 8.3 sits at the 9.3 level and the next at the 8.8 level.")

In [ ]:
f = ptx_files[ptx_files.exp == "903_046"]
for j in [j for j in ptx_jobs if j["path"].startswith("903_046")][:3]:
    print(j["path"], "channel names", C.header_info(j["path"])["channels"][:4])
print(f[["file", "PD_channel", "LP_channel", "PD_pdn_lag_ms", "PD_pdn_match", "LP_pdn_lag_ms", "LP_pdn_match", "pdn_chance"]].round(3).to_string(index=False))
r = f.iloc[0]
print(f"{r.file}: the input named {r.PD_channel} is followed by a pdn spike {r.PD_pdn_lag_ms:.0f} ms later for {r.PD_pdn_match:.2f} of its spikes, "
      f"the input named {r.LP_channel} for {r.LP_pdn_match:.2f}, against {r.pdn_chance:.3f} by chance; in the other files PD3 is the one that matches.")

In [ ]:
from collections import Counter
print("file types in the project:", dict(Counter(p.rsplit(".", 1)[-1] for p in C.FILES)))
print("files other than .abf:", sorted({p.split("/")[-1] for p in C.FILES if not p.endswith(".abf")}))
print("No state annotation (stateData_STG.xlsx, stateData_CG.xlsx), Spike2 burst output (*_bursts.txt) or gastric annotation is in the project.")

In [ ]:
pl = plans["877_101"]
print("877_101 electrodes:", pl["desc"]["electrodes"], "; file channels:", pl["heads"][pl["seq"][0][0]]["channels"])
print("No pdn, and the intracellular channel ME1_Vx is not named in the README: the PD unit cannot be isolated by threshold.")

## Extracellular units: spikes, bursts and the state proxy

Each unit's threshold, burst rule and the proxy for rhythmic activity are in `crabph` (its docstring gives the source of each parameter). The table gives, per unit, the threshold in its first control file, the median spike height there, and the median burst period at control. A step an experiment lacks is 0 in the main reading, as the paper's code gives it (a condition marked missing in `extracellularAnalysis.m` has no bursts, and `pHCumulative.m` sets an empty bin to 0; its state is NaN, which counts as no time in any state), and is left out in the variant.

In [ ]:
units = C.ext_units(plans, ext_results)
U = {u["label"]: u for u in units}
rows = []
for u in units:
    if u["out"] is None:
        rows.append(dict(unit=u["label"], ganglion=u["ganglion"], channel=u["channel"], note="no channel")); continue
    ref_col = next(c for c in ("ctl_acid", "ctl_base") if u["out"][c] and u["out"][c]["file"] == plans[u["exp"]]["seq"][0][0])
    v = u["out"][ref_col]
    rows.append(dict(unit=u["label"], ganglion=u["ganglion"], channel=u["channel"], order=u["order"], thr_control=round(v["thr"], 4),
                     noise_sd=round(v["sd"], 4), median_spike_height=round(u["ref"], 4), control_period_s=round(u["ctl_period"], 3),
                     control_bursts=len(v["bursts"])))
print(pd.DataFrame(rows).to_string(index=False))
measures = {(g, m, miss): C.measure_frame(units, g, m, missing=miss) for g in ("STG", "CG") for m in ("hz", "spikes", "duty") for miss in ("zero", "nan")}
code_measures = {(g, m): C.measure_frame(units, g, m, window="code") for g in ("STG", "CG") for m in ("hz", "spikes", "duty")}
rhythm = {(g, miss): C.rhythmic_frame(units, g, missing=miss) for g in ("STG", "CG") for miss in ("zero", "nan")}
for g in ("STG", "CG"):
    for m in ("hz", "spikes", "duty"):
        df = measures[(g, m, "zero")]
        print(f"{g} {m}: range {df.values.min():.3f} to {df.values.max():.3f}; preparations {len(df)}")
        print(df.round(3).to_string())
    print(f"{g} fraction of the last 8 min rhythmic (proxy)")
    print(rhythm[(g, "zero")].round(2).to_string())

The state readings and their parameters. The paper's states were annotated by hand from all units (STG: PD, LP and PY; CG: Small and Large Cells), and the annotations are not in the dataset, so the claims file asks for states annotated from its definitions ([STATE]). They are read here from the spikes, with choices of this attempt (assumptions):

- CG state 1 (SC and LC bursting): a second lies between two LC burst onsets no more than `RHYTHM_GAP_FACTOR` (3) control periods apart, with runs under 10 s merged (Methods, paragraph 3). The LC fires with the SC bursts, so this is close to state 1; it cannot tell state 2 (SC only) from state 3 (silent).
- STG states 1 and 2 (normal or weak triphasic): the same rule on the PD bursts of the pdn, and in addition each PD cycle must hold at least `MIN_OTHER_SPIKES` (2) lvn spikes outside the PD bursts, the LP and PY firing that states 1 and 2 require (`crabph.triphasic_seconds`). A PD rhythm with LP and PY silent (state 3, intermittent triphasic) is not counted. LP and PY are not told apart, so this reading cannot separate state 1 from state 2, or a cycle where only one of them fires from one where both do.

Variants shown for the claims that use them: gap factors 2 and 5; for the STG, 1 and 4 lvn spikes per cycle, and PD bursting alone (the reading of the first round, which counts state 3 as rhythmic).

In [ ]:
def proxy_frames(factor):
    old = C.RHYTHM_GAP_FACTOR
    C.RHYTHM_GAP_FACTOR = factor
    try:
        us = C.ext_units(plans, ext_results)
    finally:
        C.RHYTHM_GAP_FACTOR = old
    return {g: C.rhythmic_frame(us, g) for g in ("STG", "CG")}
proxy_var = {f: proxy_frames(f) for f in (2.0, 5.0)}
proxy_var[3.0] = {g: rhythm[(g, "zero")] for g in ("STG", "CG")}
for f in (2.0, 3.0, 5.0):
    print(f"factor {f}: mean fraction rhythmic per condition")
    print(pd.DataFrame({g: proxy_var[f][g].mean() for g in ("STG", "CG")}).T.round(3).to_string())
units_cg03 = C.ext_units(plans, ext_results, cg_rule=(C.MAX_ISI_S, C.MIN_SPIKES))
cg03 = {m: C.measure_frame(units_cg03, "CG", m) for m in ("hz", "spikes", "duty")}
cg03_rhythm = C.rhythmic_frame(units_cg03, "CG")
stg_pd = {miss: C.rhythmic_frame(units, "STG", missing=miss, key="rhythmic_pd") for miss in ("zero", "nan")}
stg_plans = {e: pl for e, pl in plans.items() if any(g == "STG" for g, _ in pl["units"].values())}
def other_frames(k):
    old = C.MIN_OTHER_SPIKES
    C.MIN_OTHER_SPIKES = k
    try:
        us = C.ext_units(stg_plans, ext_results)
    finally:
        C.MIN_OTHER_SPIKES = old
    return C.rhythmic_frame(us, "STG")
stg_other = {k: other_frames(k) for k in (1, 4)}
stg_other[2] = rhythm[("STG", "zero")]
print("STG lvn channel per experiment:", {e: pl["lvn"] for e, pl in stg_plans.items()})
print("STG mean fraction rhythmic per condition, by reading:")
print(pd.DataFrame({"lvn, 2 spikes (main)": stg_other[2].mean(), "lvn, 1 spike": stg_other[1].mean(), "lvn, 4 spikes": stg_other[4].mean(),
                    "PD bursting alone": stg_pd["zero"].mean()}).T.round(3).to_string())
d = (stg_pd["zero"] - rhythm[("STG", "zero")]).round(2)
print("PD-alone fraction minus the lvn reading, per preparation (nonzero rows):")
print(d[(d != 0).any(axis=1)].to_string())

LC isolation check. For every CG unit, the events of its trunk channel (high-passed, above 4 noise SDs, last 8 min) at the controls, 5.5 and 10.4, counted in bins of height relative to the file's threshold (a text histogram: the columns below 1.0 are not counted as LC spikes). `count_med_over_ctl` is the median height of the counted spikes over the unit's control median: near 1 when the same unit is counted, well below 1 when smaller units (Small Cells) cross the threshold. `single` is the fraction of bursts with one spike. The four units the verifier named are marked.

In [ ]:
flag = ["897_041 CG3L", "887_149 CG2L", "877_149 CG1L", "877_149 CG2L"]
edges = np.array([0, 0.5, 0.75, 1.0, 1.5, 2, 3, 5, np.inf])
rows = []
for u in units:
    if u["ganglion"] != "CG" or u["out"] is None:
        continue
    for col in ("ctl_acid", "ctl_base", "5.5", "10.4"):
        v = u["out"][col]
        if v is None:
            continue
        h = R[v["file"]]["ext"][u["channel"]]["late"][1]
        hist = np.histogram(h / v["thr"], bins=edges)[0]
        b = v["bursts"]
        rows.append(dict(unit=u["label"] + (" *" if u["label"] in flag else ""), cond=col, thr_over_sd=round(float(v["thr"] / v["sd"]), 1),
                         counted=v["n_spikes"], bursts=len(b), single=round(float((b.spikes == 1).mean()), 2) if len(b) else np.nan,
                         spikes_per_burst=round(float(b.spikes.median()), 1) if len(b) else np.nan,
                         count_med_over_ctl=round(float(np.median(h[h >= v["thr"]]) / u["ref"]), 2) if (h >= v["thr"]).any() else np.nan,
                         **{f"{lo:g}-{hi:g}": n for lo, hi, n in zip(edges[:-1], edges[1:], hist)}))
iso = pd.DataFrame(rows)
print(iso.to_string(index=False))
print("CG measures without the four marked units (a variant):")
keep = [l for l in rhythm[("CG", "zero")].index if l not in flag]
def cg_without(kept):
    for name, df in [("fraction rhythmic", rhythm[("CG", "zero")]), ("frequency", measures[("CG", "hz", "zero")]),
                     ("spikes per burst", measures[("CG", "spikes", "zero")]), ("duty cycle", measures[("CG", "duty", "zero")])]:
        a, b, t = C.rm_report(df.loc[kept], "", show=False)
        print(f"  {name}: n = {len(kept)}; acid ANOVA p = {a['p']:.3g}, base p = {b['p']:.3g}; corrected p and diff at",
              {s: (round(t.loc[s, 'diff'], 3), float(f"{t.loc[s, 'p_corr']:.3g}")) for s in ("5.5", "6.1", "9.3", "9.8", "10.4")})
cg_without(keep)
# three more units with one step of doubtful isolation in the table above (verifier, round 2)
more = ["887_127 CG1L", "897_041 CG1L", "887_005 CG2L"]
keep7 = [l for l in keep if l not in more]
print("CG measures without the four marked units and", more, "(a variant):")
cg_without(keep7)

## fig2a-stg1-traces

> In the example acid-first STG preparation, the pyloric rhythm stayed triphasic in acid until the most acidic pH, 5.5; in base it stayed triphasic up to 10.4 with fewer spikes per burst and smaller slow waves at 10.4, and it recovered in control saline after each protocol.

Criterion: on 897_037, state 1 or 2 for most of each step from 7.2 to 6.1 and 8.3 to 10.4 and each recovery; at 10.4 lower PD spikes per burst ([EXT]) and lower PD slow-wave amplitude ([INTRA] filtering, last minute) than at the control before the base steps. States by the proxy (assumption). The PD intracellular channel is VM1_2 (README "Vm1_2: PD").

In [ ]:
ex = C.STG_EXAMPLE
fr = rhythm[("STG", "zero")].loc[ex]
print("order", plans[ex]["order"], "; fraction rhythmic (proxy) per condition:"); print(fr.round(3).to_string())
need = ["7.2", "6.7", "6.1", "8.3", "8.8", "9.3", "9.8", "10.4", "rec_acid", "rec_base"]
ok_states = all(fr[c] > 0.5 for c in need)
print("most of each step 7.2 to 6.1, 8.3 to 10.4 and both recoveries rhythmic:", ok_states, "; at 5.5:", round(fr["5.5"], 3))
for f in (2.0, 5.0):
    print(f"  proxy factor {f}:", {c: round(proxy_var[f]['STG'].loc[ex, c], 2) for c in need + ['5.5']})
for k in (1, 4):
    print(f"  {k} lvn spikes per cycle:", {c: round(stg_other[k].loc[ex, c], 2) for c in need + ['5.5']})
print("  PD bursting alone:", {c: round(stg_pd['zero'].loc[ex, c], 2) for c in need + ['5.5']})
sp = measures[("STG", "spikes", "zero")].loc[ex]
print(f"PD spikes per burst: control before base {sp['ctl_base']:.2f}, 10.4 {sp['10.4']:.2f}")
intact = C.intact_intra_frames(plans, ext_results)
wa = intact[("PD", "wave_amp")].loc[ex]
print(f"PD slow-wave amplitude (mV, last minute): control before base {wa['ctl_base']:.2f}, 10.4 {wa['10.4']:.2f}")
print("all parts hold:", ok_states and sp["10.4"] < sp["ctl_base"] and wa["10.4"] < wa["ctl_base"])

**Result: reproduced, on the STG state reading (assumed).** 897_037 is rhythmic over the whole of every step from 7.2 to 6.1 and from 8.3 to 10.4 and in both recoveries, and also at 5.5 (reading (a)), with PD bursting and lvn firing between the PD bursts in every cycle. The same holds under gap factors 2 and 5, with 1 or 4 lvn spikes per cycle, and with PD bursting alone. At 10.4 PD spikes per burst (12.43 against 13.10) and the PD slow-wave amplitude (17.65 against 18.03 mV) are lower than at the control before the base steps, both by small margins. LP and PY are not told apart on the lvn, so "triphasic" rests on PD bursting with lvn firing between the bursts. The lvn condition hardly separates the readings: across all STG preparations it changes the fraction rhythmic only at 5.5 in 877_121 (by 0.19) and 887_049 (by 0.04) against PD bursting alone, so this reading rests mainly on PD bursting.

## fig2bcd-stg1-measures

> In the example STG preparation, pyloric burst frequency increased in acid and decreased in base, PD spikes per burst decreased at pH 5.5, and PD duty cycle decreased slightly in base.

Criterion: 897_037, 48 bin means per step; reading (a) the extreme step against the control before that protocol, (b) the sign of the Spearman correlation of the step means with pH over the control and the steps.

In [ ]:
def example_readings(g, label, frames):
    hz, spk, duty = (frames[m].loc[label] for m in ("hz", "spikes", "duty"))
    acid_cols, base_cols = ["ctl_acid"] + C.ACID_STEPS, ["ctl_base"] + C.BASE_STEPS
    out = {}
    out["a"] = dict(hz_acid=hz["5.5"] - hz["ctl_acid"], hz_base=hz["10.4"] - hz["ctl_base"],
                    spikes_acid=spk["5.5"] - spk["ctl_acid"], duty_acid=duty["5.5"] - duty["ctl_acid"],
                    duty_base=duty["10.4"] - duty["ctl_base"], spikes_base=spk["10.4"] - spk["ctl_base"])
    sp = lambda s, cols: C.spearman_sign(s, cols, C.PH_OF)
    # Spearman with pH: a negative value means the measure rises as pH falls (acid) or falls as pH rises (base)
    out["b"] = dict(hz_acid=sp(hz, acid_cols), hz_base=sp(hz, base_cols), spikes_acid=sp(spk, acid_cols),
                    duty_acid=sp(duty, acid_cols), duty_base=sp(duty, base_cols), spikes_base=sp(spk, base_cols))
    return pd.DataFrame(out).round(3)

stg_frames = {m: measures[("STG", m, "zero")] for m in ("hz", "spikes", "duty")}
r = example_readings("STG", ex, stg_frames)
print("(a) extreme minus control; (b) Spearman rho of step means with pH")
print(r.to_string())
print("step means:"); print(pd.DataFrame({m: stg_frames[m].loc[ex] for m in stg_frames}).T.round(3).to_string())
ok_a = r.loc["hz_acid", "a"] > 0 and r.loc["hz_base", "a"] < 0 and r.loc["spikes_acid", "a"] < 0 and r.loc["duty_base", "a"] < 0
# (b): a rise in acid is a negative rho with pH over the acid steps; a fall in base is a negative rho over the base steps
ok_b = r.loc["hz_acid", "b"] < 0 and r.loc["hz_base", "b"] < 0 and r.loc["spikes_acid", "a"] < 0 and r.loc["duty_base", "b"] < 0
print("reading (a) holds:", ok_a, "; reading (b) holds:", ok_b, "(spikes at 5.5 against control is the same in both)")
code_frames = {m: code_measures[("STG", m)] for m in ("hz", "spikes", "duty")}
print("code window variant:"); print(example_readings("STG", ex, code_frames).to_string())

**Result: reproduced (assumed: automatic spike thresholds).** In 897_037 frequency is higher at 5.5 than at its control (+0.118 Hz) and lower at 10.4 (-0.266 Hz); its Spearman rho with pH is -0.80 over the acid steps and -1.00 over the base steps. PD spikes per burst at 5.5 are 5.40 below control, and duty cycle at 10.4 is 0.048 below control (rho -0.94 over the base steps). Both readings hold. In the code's window the frequency and spike parts hold, and duty cycle in base falls by 0.023 (rho -0.54).

## fig3a-stg-variability-examples

Blocked (`metadata_insufficient`). The criterion needs the STG states 1 (normal) and 2 (weak triphasic) apart at 10.4, and in state 2, 3 or 4 at 5.5. The manual annotation is not in the dataset, and the reading of the states from the pdn and lvn cannot separate state 1 from state 2, as LP and PY are not told apart. The parts that can be read are shown below; they do not decide the claim.

In [ ]:
# the parts of the fig3a criterion this attempt can read: rhythmic (states 1 or 2) for most of every step from 6.1 to 9.8, not at 5.5;
# the state at 10.4 (1 against 2) cannot be read, so only the fraction and the frequency at 10.4 are shown
st = rhythm[("STG", "zero")]
mid = ["6.1", "6.7", "7.2", "ctl_acid", "ctl_base", "8.3", "8.8", "9.3", "9.8"]
cand = st[(st[mid] > 0.5).all(axis=1) & (st["5.5"] <= 0.5)].index
hz = measures[("STG", "hz", "zero")]
print("STG preparations other than 897_037 rhythmic for most of 6.1 to 9.8 and not at 5.5:", [c for c in cand if c != C.STG_EXAMPLE])
print(pd.DataFrame({"frac_5.5": st.loc[cand, "5.5"], "frac_10.4": st.loc[cand, "10.4"], "hz_10.4_minus_ctl": hz.loc[cand, "10.4"] - hz.loc[cand, "ctl_base"],
                    "has_5.5_file": [U[c]["out"]["5.5"] is not None for c in cand],
                    "has_10.4_file": [U[c]["out"]["10.4"] is not None for c in cand]}).round(3).to_string())

Four preparations besides 897_037 are rhythmic for most of 6.1 to 9.8 and not at 5.5 (887_005, 887_097, 887_105, 887_145). 887_005 has no 10.4 file, so its fraction 0 there is the zero-fill of a missing step, not a recorded silence; the other three are rhythmic throughout (fraction 1.0) with frequency 0.14 to 0.29 Hz below control. Whether one of these is in state 1 and another in state 2 at 10.4 cannot be read, so the claim stays blocked.

## fig3c-stg-rhythmic-fraction

> Both acid and base significantly decreased the fraction of time STG preparations were rhythmic (normal or weak triphasic), and the rhythm was significantly less triphasic at pH 5.5 and 10.4 than at control.

Criterion: per-preparation fraction rhythmic over the last 8 min, [RM]; both ANOVAs p < 0.05 and corrected t-tests at 5.5 and 10.4 p < 0.05, lower. Fraction rhythmic from the proxy (assumption).

In [ ]:
def rm_claim(df, label):
    a, b, t = C.rm_report(df, label)
    return a, b, t
print("STG preparations with the proxy:", len(rhythm[("STG", "zero")]))
a, b, t = rm_claim(rhythm[("STG", "zero")], "STG fraction rhythmic, missing steps as 0")
fig3c = dict(a=a, b=b, t=t)
print("criterion:", a["p"] < 0.05, b["p"] < 0.05, t.loc["5.5", "p_corr"] < 0.05 and t.loc["5.5", "diff"] < 0, t.loc["10.4", "p_corr"] < 0.05 and t.loc["10.4", "diff"] < 0)
C.rm_report(rhythm[("STG", "nan")], "variant: missing steps left out")
for f in (2.0, 5.0):
    C.rm_report(proxy_var[f]["STG"], f"variant: proxy factor {f}")
for k in (1, 4):
    C.rm_report(stg_other[k], f"variant: {k} lvn spikes per cycle")
_ = C.rm_report(stg_pd["zero"], "variant: PD bursting alone (counts state 3)")

**Result: not reproduced, on the STG state reading (assumed).** With 13 STG preparations, both ANOVAs are significant (acid p = 3.9e-08, base p = 0.021), and the corrected t-test at 5.5 is significant (fraction 0.375 against 1.0, p = 9.0e-04). The one at 10.4 is not (0.771, p = 0.31), and the criterion needs both. The expected 15 preparations are 13 here (tolerance 0). The same split holds with missing steps left out (10.4: p = 0.61), with gap factors 2 and 5 (p = 0.22 and 0.41), with 1 or 4 lvn spikes per cycle (p = 0.31) and with PD bursting alone (p = 0.31).

## stg-state-transitions

Blocked (`metadata_insufficient`). The criterion counts transitions among the four STG states over whole recordings; the annotation is not in the dataset, and the proxy has two states.

## fig3d-stg-frequency

> Mean pyloric burst frequency was relatively invariant across acid steps but varied significantly across base steps, being significantly slower than control at pH 9.3 and 10.4.

Criterion: [RM] on PD burst frequency; acid ANOVA p ≥ 0.05 (absence), base ANOVA p < 0.05, corrected t-tests at 9.3 and 10.4 p < 0.05, lower.

In [ ]:
print("STG preparations:", len(measures[("STG", "hz", "zero")]), "(877_101 has no pdn; 877_127 is not in the project)")
a, b, t = rm_claim(measures[("STG", "hz", "zero")], "PD burst frequency (Hz), missing steps as 0")
print("criterion: acid p >= 0.05:", a["p"] >= 0.05, "; base p < 0.05:", b["p"] < 0.05,
      "; 9.3:", t.loc["9.3", "p_corr"] < 0.05 and t.loc["9.3", "diff"] < 0, "; 10.4:", t.loc["10.4", "p_corr"] < 0.05 and t.loc["10.4", "diff"] < 0)
C.rm_report(measures[("STG", "hz", "nan")], "variant: missing steps left out")
C.rm_report(code_measures[("STG", "hz")], "variant: the code's window (480 s from the first burst)")
df = measures[("STG", "hz", "zero")]
C.rm_report(df[df["5.5"] > 0], "variant: preparations still bursting at 5.5 only")

**Result: not reproduced (assumed: automatic spike thresholds).** The base part holds in part: base ANOVA p = 4.4e-04, and frequency at 10.4 is lower than control (-0.407 Hz, corrected p = 0.047), while 9.3 is not significant (-0.206 Hz, p = 0.65). The acid ANOVA is significant (p = 1.6e-07), driven by 5.5 (-0.837 Hz, p = 0.0011), where several preparations fall silent and their empty bins count as 0. The claim of absence in acid therefore fails under the Methods window. In the code's window (480 s from the first burst of each file), the acid ANOVA gives p = 0.061, as the paper reports, but no base t-test is significant there. The criterion has no partial condition. Whether it reads "relatively invariant" as the paper meant is asked of a reader (questions/haley-2018.yaml).

## fig3e-stg-spikes-per-burst

> Acid and base both significantly affected the number of PD spikes per burst, which was significantly reduced at pH 5.5.

In [ ]:
a, b, t = rm_claim(measures[("STG", "spikes", "zero")], "PD spikes per burst, missing steps as 0")
print("criterion:", a["p"] < 0.05, b["p"] < 0.05, t.loc["5.5", "p_corr"] < 0.05 and t.loc["5.5", "diff"] < 0)
C.rm_report(measures[("STG", "spikes", "nan")], "variant: missing steps left out")
C.rm_report(code_measures[("STG", "spikes")], "variant: the code's window")

**Result: reproduced (assumed: automatic spike thresholds).** Acid ANOVA p = 1.6e-11 and base ANOVA p = 1.0e-06 (13 preparations). PD spikes per burst at 5.5 are 2.24 against 9.43 at control (corrected p = 3.2e-05). The result holds with missing steps left out and in the code's window.

## fig3f-stg-duty-cycle

> Acid and base both significantly affected PD duty cycle, with a slight increase from control pH 7.8 to pH 6.1.

In [ ]:
a, b, t = rm_claim(measures[("STG", "duty", "zero")], "PD duty cycle, missing steps as 0")
print("criterion:", a["p"] < 0.05, b["p"] < 0.05, t.loc["6.1", "p_corr"] < 0.05 and t.loc["6.1", "diff"] > 0)
C.rm_report(measures[("STG", "duty", "nan")], "variant: missing steps left out")
C.rm_report(code_measures[("STG", "duty")], "variant: the code's window")

**Result: not reproduced (assumed: automatic spike thresholds).** Both ANOVAs are significant (acid p = 2.3e-09, base p = 3.6e-10). The duty cycle at 6.1 is 0.023 lower than control, not higher (corrected p = 0.93). In the code's window it is 0.041 higher at 6.1 (p = 0.09). The criterion has no partial condition.

## fig3s2-stg-control-variability

> Pyloric frequency, PD spikes per burst and PD duty cycle vary widely between the 15 STG preparations at control pH, as widely as the pooled distributions across pH.

Criterion: the 48 bins of each preparation in its first control step (condition 1 of the code); (a) the range of the preparation means at control spans at least half the range of the pooled bin values over all steps, for each measure; (b) Kruskal-Wallis of the bins on preparation p < 0.05 for each measure.

In [ ]:
def control_variability(g):
    out = {}
    for m in ("hz", "spikes", "duty"):
        ctl_bins, pooled = [], []
        for u in units:
            if u["ganglion"] != g or u["out"] is None:
                continue
            first = plans[u["exp"]]["seq"][0][0]
            col = next(c for c in ("ctl_acid", "ctl_base") if u["out"][c] and u["out"][c]["file"] == first)
            ctl_bins.append(u["out"][col]["bins"][m])
            for c in C.COLUMNS[1:5] + ["ctl_acid", "ctl_base"] + C.COLUMNS[7:12]:
                if u["out"][c] is not None:
                    pooled.append(u["out"][c]["bins"][m])
        means = np.array([b.mean() for b in ctl_bins])
        pooled = np.concatenate(pooled)
        kw = stats.kruskal(*ctl_bins)
        out[m] = dict(n=len(ctl_bins), prep_mean_min=means.min(), prep_mean_max=means.max(),
                      prep_range=np.ptp(means), pooled_min=pooled.min(), pooled_max=pooled.max(), pooled_range=np.ptp(pooled),
                      ratio=np.ptp(means) / np.ptp(pooled), reading_a=np.ptp(means) >= 0.5 * np.ptp(pooled),
                      kruskal_H=kw.statistic, kruskal_p=kw.pvalue, reading_b=kw.pvalue < 0.05)
    return pd.DataFrame(out).T
v = control_variability("STG")
print(v.to_string())
print("reading (a) for all three:", bool(v.reading_a.all()), "; reading (b) for all three:", bool(v.reading_b.all()))

**Result: reproduced under reading (b) (assumed: automatic spike thresholds).** At the first control step, a Kruskal-Wallis test of the 48 bins on preparation gives H = 606, 545 and 491 for frequency, spikes per burst and duty cycle (13 preparations; p prints as 0, below double precision). Under reading (a), the range of the preparation means is 0.48, 0.75 and 0.37 of the pooled range for the three measures, so (a) holds for spikes per burst only.

## fig4a-ptx-example

> In an example preparation with PTX, the PD neuron lost most of its slow wave at pH 6.7, became silent and depolarized at 5.5, kept its burst at 8.8 and fired single-spike bursts at 10.4; the LP neuron fired tonically from 6.7 to 10.4 and lost activity at 5.5.

Criterion: at least one PTX preparation with PD slow-wave amplitude at 6.7 below half its PTX control, no PD spikes and minimum Vm above control at 5.5, one spike per PD burst at 10.4, LP spiking without slow-wave bursts from 6.7 to 10.4, no LP spikes at 5.5. Readings of the words taken here: "one spike per PD burst" is a ratio of PD spikes to slow-wave peaks of at most 1.5 (and above 0), and "without slow-wave bursts" is LP firing with an interspike-interval coefficient of variation below 1 (tonic firing, as opposed to bursts of spikes separated by long gaps). The PTX control is the control before the acid steps.

In [ ]:
rows = []
for e in C.PTX_EXPERIMENTS:
    g = lambda u, k, c: ptx[(u, k)].loc[e, c]
    ratio104 = g("PD", "n_spikes", "10.4") / g("PD", "n_waves", "10.4") if g("PD", "n_waves", "10.4") else np.nan
    lp_tonic = [(g("LP", "n_spikes", c) > 0) and (g("LP", "isi_cv", c) < 1) for c in ["6.7", "8.3", "8.8", "9.3", "9.8", "10.4"]]
    rows.append(dict(exp=e,
                     pd_wave_67_over_ctl=g("PD", "wave_amp", "6.7") / g("PD", "wave_amp", "ctl_acid"),
                     pd_spikes_55=g("PD", "n_spikes", "5.5"), pd_vmin_55_minus_ctl=g("PD", "vmin_methods", "5.5") - g("PD", "vmin_methods", "ctl_acid"),
                     pd_spikes_per_wave_104=ratio104, pd_wave_88_over_ctl=g("PD", "wave_amp", "8.8") / g("PD", "wave_amp", "ctl_base"),
                     lp_tonic_67_to_104=f"{sum(lp_tonic)}/6", lp_spikes_55=g("LP", "n_spikes", "5.5")))
fa = pd.DataFrame(rows)
fa["all"] = ((fa.pd_wave_67_over_ctl < 0.5) & (fa.pd_spikes_55 == 0) & (fa.pd_vmin_55_minus_ctl > 0) & (fa.pd_spikes_per_wave_104 > 0)
             & (fa.pd_spikes_per_wave_104 <= 1.5) & (fa.lp_tonic_67_to_104 == "6/6") & (fa.lp_spikes_55 == 0))
print(fa.round(3).to_string(index=False))
print("PTX preparations meeting every part:", int(fa["all"].sum()))
parts = {"PD wave at 6.7 < half": (fa.pd_wave_67_over_ctl < 0.5), "PD silent at 5.5": fa.pd_spikes_55 == 0, "PD depolarized at 5.5": fa.pd_vmin_55_minus_ctl > 0,
         "one spike per PD burst at 10.4": (fa.pd_spikes_per_wave_104 > 0) & (fa.pd_spikes_per_wave_104 <= 1.5),
         "LP tonic 6.7 to 10.4": fa.lp_tonic_67_to_104 == "6/6", "LP silent at 5.5": fa.lp_spikes_55 == 0}
print({k: int(v.sum()) for k, v in parts.items()})
print("parts met per preparation:", dict(zip(fa.exp, sum(v.astype(int) for v in parts.values()))))
print("PD label check of 903_052 (lag of the pdn spikes after each intracellular spike; fraction followed at that lag; chance):")
print(ptx_files[ptx_files.exp == "903_052"][["file", "label", "PD_channel", "PD_pdn_lag_ms", "PD_pdn_match", "LP_pdn_lag_ms", "LP_pdn_match", "pdn_chance"]].round(3).to_string(index=False))

**Result: reproduced (assumed: the readings of "one spike per burst" and "tonic" named above).** One PTX preparation, 903_052, meets every part. Its PD slow wave at 6.7 is 0.41 of control, and at 5.5 PD is silent and depolarized by 16.4 mV. At 10.4 it fires 1.0 spikes per slow wave, and LP fires tonically at all six steps from 6.7 to 10.4 and is silent at 5.5. Every preparation has a silent, depolarized PD at 5.5, while the other parts hold in two to seven of the nine. The PD label of 903_052 holds: in 7 of its 9 files with PD spikes, 0.61 to 1.0 of them are followed by a pdn spike 34 to 38 ms later, against 0.015 to 0.037 by chance, while LP spikes are followed at about chance (0.02 to 0.07). The other two are the 6.7 file (0.02) and the last recovery (0.21). The first round's check used a 5 ms window, which this conduction delay falls outside. The outcome rests on this one preparation. Two caveats. First, 903_052's bath electrode responds less than the others (slope -37.6 mV per pH unit, against -45.2 to -46.7 in the other eight), so its "10.4" and "5.5" files read as pH 9.75 and 5.87 on the common slope. The response is linear over all its files (r = -0.999), as in the others, which fits an electrode of lower sensitivity better than two solutions that were off; the data cannot exclude the latter, and the labels follow the rank of the signal within each complete run. Second, in the 6.7 file, which the slow-wave part uses, PD spikes match the pdn at 0.02 against 0.011 by chance, so PD identity is not confirmed in that file. It rests on the intracellular electrode staying in the same cell, as the files before it (7.2, 0.61) and after it (the last recovery, 0.21 against 0.008) indicate.

## fig4bc-membrane-potential

> With PTX, the PD neuron depolarized significantly in extreme acid (pH 5.5 and 6.1) while LP stayed relatively constant, and in base PD stayed relatively constant while LP hyperpolarized significantly at pH 8.8, 9.3, 9.8 and 10.4.

Criterion: minimum membrane potential ([INTRA]; Methods reading, the minimum of the unfiltered Vm between bursts), [RM]. A PTX preparation lacking a step is left out of that protocol's ANOVA and of that t-test (the paper's PTX code has no rule for a missing step).

In [ ]:
res4bc = {}
for k in ("vmin_methods", "vmin_code"):
    print("==", k)
    pa, pb, pt = C.rm_report(ptx[("PD", k)], f"PD minimum Vm ({k})")
    la, lb, lt = C.rm_report(ptx[("LP", k)], f"LP minimum Vm ({k})")
    tests = {"PD 5.5": pt.loc["5.5", "p_corr"] < 0.05 and pt.loc["5.5", "diff"] > 0, "PD 6.1": pt.loc["6.1", "p_corr"] < 0.05 and pt.loc["6.1", "diff"] > 0}
    for s in ("8.8", "9.3", "9.8", "10.4"):
        tests[f"LP {s}"] = lt.loc[s, "p_corr"] < 0.05 and lt.loc[s, "diff"] < 0
    res4bc[k] = dict(pd_acid=pa["p"], lp_base=lb["p"], lp_acid=la["p"], pd_base=pb["p"], tests=tests)
    print("PD acid ANOVA p < 0.05:", pa["p"] < 0.05, "; LP base ANOVA p < 0.05:", lb["p"] < 0.05, "; t-tests:", tests,
          f"; absence: LP acid ANOVA p = {la['p']:.3g}, PD base ANOVA p = {pb['p']:.3g}")
# 903_056's 8.8 file lasts 2.3 min, so its last minute is early in the step; the LP 8.8 test without it
lv = ptx[("LP", "vmin_methods")].drop(index="903_056")
d = (lv["8.8"] - lv["ctl_base"]).dropna()
tt = stats.ttest_rel(lv.loc[d.index, "8.8"], lv.loc[d.index, "ctl_base"])
print(f"variant without 903_056: LP 8.8 minus control {d.mean():.2f} mV, n = {len(d)}, corrected p = {min(1, tt.pvalue * 5):.3g}")
_, lb2, _ = C.rm_report(lv, "variant without 903_056: LP minimum Vm", show=False)
print(f"   LP base ANOVA without 903_056: p = {lb2['p']:.3g}")

**Result: reproduced (assumed: the code's starting parameters for the waveform analysis and the conditions of the PTX files taken from their recorded pH).** PD acid ANOVA p = 6.4e-10 and LP base ANOVA p = 8.9e-14 (8 complete preparations of 9). PD is depolarized at 5.5 (+19.0 mV, corrected p = 7.9e-07) and 6.1 (+11.6 mV, p = 0.016). LP is hyperpolarized at 8.8 (-2.2 mV, p = 0.020), 9.3 (-9.5 mV, p = 0.0025), 9.8 (-12.1 mV, p = 3.5e-04) and 10.4 (-12.6 mV, p = 2.0e-04). The parts claimed relatively constant are not significant: LP acid ANOVA p = 0.65 and PD base ANOVA p = 0.13. The code's slow-wave troughs give the same outcome. The LP 8.8 test depends on 903_056, whose 8.8 file lasts 2.3 min: without it LP 8.8 is -2.1 mV with corrected p = 0.056, which would make the outcome partial (five of six t-tests).

## fig4de-spike-amplitude

> With PTX, acid significantly affected spike amplitude of both PD and LP, and base affected LP but not PD; amplitude was significantly reduced at pH 5.5 in both, and LP amplitude also at pH 6.1.

In [ ]:
pa, pb, pt = C.rm_report(ptx[("PD", "spike_amp")], "PD spike amplitude (mV)")
la, lb, lt = C.rm_report(ptx[("LP", "spike_amp")], "LP spike amplitude (mV)")
crit = {"PD acid": pa["p"] < 0.05, "LP acid": la["p"] < 0.05, "LP base": lb["p"] < 0.05, "PD base absent": pb["p"] >= 0.05,
        "PD 5.5": pt.loc["5.5", "p_corr"] < 0.05 and pt.loc["5.5", "diff"] < 0, "LP 5.5": lt.loc["5.5", "p_corr"] < 0.05 and lt.loc["5.5", "diff"] < 0,
        "LP 6.1": lt.loc["6.1", "p_corr"] < 0.05 and lt.loc["6.1", "diff"] < 0}
print(crit)

**Result: partially reproduced (assumed, as fig4bc).** The acid ANOVAs are significant for PD (p = 1.5e-06) and LP (p = 0.0015), and the PD base ANOVA is not (p = 0.26), as claimed. Amplitude at 5.5 is lower for PD (corrected p = 7.8e-04) and LP (p = 0.0044). The LP base ANOVA is not significant (p = 0.095), and the fall in LP amplitude at 6.1 (-3.4 mV) is not significant (p = 0.11).

## fig4fg-ptx-frequency

> With PTX, acid and base both significantly affected PD burst frequency and LP firing rate; PD burst frequency was significantly lower at pH 5.5, and LP firing rate was significantly lower at pH 5.5, 6.1, 8.3, 9.3, 9.8 and 10.4 than at control.

PD burst frequency from slow-wave trough-to-trough intervals (Methods); the code's peak-to-peak is a variant.

In [ ]:
for k in ("burst_hz_trough", "burst_hz_peak"):
    pa, pb, pt = C.rm_report(ptx[("PD", k)], f"PD burst frequency ({k})")
    la, lb, lt = C.rm_report(ptx[("LP", "spike_hz")], "LP firing rate (Hz)", show=(k == "burst_hz_trough"))
    tt = {"PD 5.5": pt.loc["5.5", "p_corr"] < 0.05 and pt.loc["5.5", "diff"] < 0}
    for s in ("5.5", "6.1", "8.3", "9.3", "9.8", "10.4"):
        tt[f"LP {s}"] = lt.loc[s, "p_corr"] < 0.05 and lt.loc[s, "diff"] < 0
    print(k, "ANOVAs < 0.05:", [pa["p"] < 0.05, pb["p"] < 0.05, la["p"] < 0.05, lb["p"] < 0.05], "; t-tests:", tt, "; held:", sum(tt.values()), "of 7")

**Result: partially reproduced (assumed, as fig4bc).** All four ANOVAs are significant: PD acid p = 7.0e-08, PD base p = 0.0016, LP acid p = 7.3e-09, LP base p = 7.3e-10. Six of the seven named t-tests hold, including PD burst frequency at 5.5 (-1.26 Hz, p = 0.0034) and LP rate at 5.5, 6.1, 9.3, 9.8 and 10.4. LP rate at 8.3 is higher than control, not lower (+2.2 Hz, p = 0.098). The code's peak-to-peak PD frequency gives the same counts.

## gastric-like-rhythm-counts, gastric-conditions-co-occur, fig5-gastric-recovery-example

Blocked (`methods_underspecified`). The paper identified gastric-like rhythms by eye (Results, "Rhythmic gastric-like activity ...", paragraphs 2 and 3) and coded them as 8.x in its manual STG annotation, which is not in the dataset. The criterion's fallback, rhythmic LG bursting alternating with DG bursting, has no period, burst or alternation thresholds in the paper or its code, and the dataset has no LG, DG, GM or MCN1 unit labels beyond the nerve names, so any detector would rest on thresholds of this attempt alone. The co-occurrence claim uses the counts, and the Figure 5 example is not named and rests on the same identification. The table below lists the nerves each STG experiment records: 8 of the 15 have an lgn (and a dgn), and 4 also an ion, so LG would have to be taken from the lvn, which it shares with PD, LP and PY, in the other 6 in the project.

In [ ]:
# which STG experiments record the nerves the gastric and state claims would need (README electrode lists)
rows = []
for e in C.STG_EXPERIMENTS:
    elec = plans[e]["desc"]["electrodes"] if e in plans else []
    nerves = [v.strip().lower() for _, v in elec]
    rows.append(dict(exp=e, in_project=e in plans, **{n: nerves.count(n) for n in ("pdn", "lvn", "pyn", "mvn", "lgn", "dgn", "ion", "son")},
                     intracellular=", ".join(v for k, v in elec if k.lower().startswith(("vm", "me")))))
el = pd.DataFrame(rows)
print(el.to_string(index=False))
print("with lgn:", int((el.lgn > 0).sum()), "; with lgn and dgn:", int(((el.lgn > 0) & (el.dgn > 0)).sum()),
      "; with lgn, dgn and ion:", int(((el.lgn > 0) & (el.dgn > 0) & (el.ion > 0)).sum()), "of", len(C.STG_EXPERIMENTS))

## fig6-cg1-measures

> In the example cardiac ganglion, the rhythm persisted over the whole pH range; cardiac frequency decreased in acid and increased in base, LC spikes per burst increased in acid, and LC duty cycle decreased slightly in acid and in base.

Criterion: 877_149 CG2L, readings (a) and (b) as for fig2bcd; and CG state 1 (proxy) for most of every step.

In [ ]:
cx = " ".join(C.CG_EXAMPLE)
cg_frames = {m: measures[("CG", m, "zero")] for m in ("hz", "spikes", "duty")}
r = example_readings("CG", cx, cg_frames)
print(r.to_string())
print("step means:"); print(pd.DataFrame({m: cg_frames[m].loc[cx] for m in cg_frames}).T.round(3).to_string())
fr = rhythm[("CG", "zero")].loc[cx]
print("fraction rhythmic (proxy):", fr.round(2).to_dict())
def cg1_ok(r, rd):
    s = 1 if rd == "a" else -1   # in (b) a rise in acid is a negative rho with pH
    if rd == "a":
        return dict(hz_acid_lower=r.loc["hz_acid", "a"] < 0, hz_base_higher=r.loc["hz_base", "a"] > 0, spikes_acid_higher=r.loc["spikes_acid", "a"] > 0,
                    duty_acid_lower=r.loc["duty_acid", "a"] < 0, duty_base_lower=r.loc["duty_base", "a"] < 0)
    return dict(hz_acid_lower=r.loc["hz_acid", "b"] > 0, hz_base_higher=r.loc["hz_base", "b"] > 0, spikes_acid_higher=r.loc["spikes_acid", "b"] < 0,
                duty_acid_lower=r.loc["duty_acid", "b"] > 0, duty_base_lower=r.loc["duty_base", "b"] < 0)
for rd in ("a", "b"):
    print("reading", rd, cg1_ok(r, rd))
print("rhythmic for most of every step:", bool((fr[C.COLUMNS] > 0.5).all()))
print("variant, CG burst rule 0.3 s and 2 spikes:"); print(example_readings("CG", cx, cg03).to_string())
print("variant, code window:"); print(example_readings("CG", cx, {m: code_measures[("CG", m)] for m in ("hz", "spikes", "duty")}).to_string())

**Result: not reproduced (assumed: automatic spike thresholds, the CG burst rule and the CG state reading).** In 877_149 CG2L the cardiac frequency is lower at 5.5 than control (-0.044 Hz) and higher at 10.4 (+0.040 Hz), and the duty cycle is lower at 5.5 (-0.133). The unit is rhythmic over every step. LC spikes per burst fall in acid (-6.05 at 5.5, rho +0.80 with pH), and the duty cycle in base does not fall (+0.018 at 10.4, rho +0.20). Both parts fail under both readings, the 0.3 s burst rule and the code's window. The criterion has no partial condition. The LC isolation check above shows about 15 to 25 spikes per burst in this unit, and its counted spikes are 3.1 times the acid control height already at the control before base, and 3.1 and 3.4 times at 5.5 and 10.4, so the isolation of this unit is in doubt.

## fig6a-cg2-less-rhythmic

> A second example cardiac ganglion became less rhythmic at pH 5.5 and at pH 9.8 and above, and recovered a normal bursting rhythm in control saline.

Criterion: among acid-first CG units other than 877_149 CG2L, one whose fraction in state 1 (proxy) is below its control at 5.5, 9.8 and 10.4, higher at each step from 6.1 to 9.3 than at each of 5.5, 9.8 and 10.4, and above half in the final recovery.

In [ ]:
def cg2_table(fr):
    rows = []
    for u in units:
        if u["ganglion"] != "CG" or u["order"] != "AB" or u["label"] == cx or u["out"] is None:
            continue
        x = fr.loc[u["label"]]
        low = [x["5.5"], x["9.8"], x["10.4"]]
        mid = [x[c] for c in ("6.1", "6.7", "7.2", "8.3", "8.8", "9.3")]
        rows.append(dict(unit=u["label"], ctl=x["ctl_acid"], f55=x["5.5"], f98=x["9.8"], f104=x["10.4"], mid_min=min(mid), rec=x["rec_base"],
                         below_ctl=all(v < x["ctl_acid"] for v in low), mid_above=min(mid) > max(low), recovered=x["rec_base"] > 0.5))
    t = pd.DataFrame(rows)
    t["all"] = t.below_ctl & t.mid_above & t.recovered
    return t
t = cg2_table(rhythm[("CG", "zero")])
print(t.round(2).to_string(index=False)); print("units meeting the criterion:", list(t.unit[t["all"]]))
for f in (2.0, 5.0):
    tv = cg2_table(proxy_var[f]["CG"]); print(f"proxy factor {f}: units meeting it:", list(tv.unit[tv["all"]]))

**Result: reproduced, on the state proxy (assumed).** Of the 8 other acid-first CG units, one, 887_093 CG2L, meets the criterion under gap factors 2, 3 and 5. Its rhythmic fraction is 0.38 at 5.5 and 0 at 9.8 and 10.4, against 1.0 at control. Every step from 6.1 to 9.3 is at least 0.63, and the final recovery is 1.0.

## fig7b-cg-rhythmic-fraction

> Acid and base both significantly affected the fraction of time cardiac preparations were rhythmic (SC and LC bursting together), which was significantly lower at pH 5.5, 9.3, 9.8 and 10.4 than at control.

In [ ]:
a, b, t = rm_claim(rhythm[("CG", "zero")], "CG fraction rhythmic (proxy), missing steps as 0")
tt = {s: t.loc[s, "p_corr"] < 0.05 and t.loc[s, "diff"] < 0 for s in ("5.5", "9.3", "9.8", "10.4")}
print("criterion: ANOVAs", a["p"] < 0.05, b["p"] < 0.05, "; t-tests", tt)
C.rm_report(rhythm[("CG", "nan")], "variant: missing steps left out")
C.rm_report(cg03_rhythm, "variant: CG burst rule 0.3 s, 2 spikes")
for f in (2.0, 5.0):
    C.rm_report(proxy_var[f]["CG"], f"variant: proxy factor {f}")

**Result: not reproduced, on the CG state reading (assumed).** The base ANOVA is significant (p = 9.9e-10), and so are the decreases at 9.3, 9.8 and 10.4 (fraction 0.55, 0.40 and 0.33 against 0.99; corrected p 0.012 to 5.9e-04). The acid ANOVA is not (p = 0.058), so neither the criterion's full nor its partial condition (both ANOVAs) is met, and [RM] runs no acid t-tests; the fall at 5.5 (0.82 against 1.0, p = 0.25) is shown for reference only. With gap factor 2 the acid ANOVA gives p = 0.023 and 5.5 is still not significant (p = 0.12), so three of the four t-tests hold and the outcome there would be partial; with factor 5 the acid ANOVA gives p = 0.29. Without the four units of doubtful isolation, both ANOVAs are significant (acid p = 0.045, base p = 1.3e-11) and so are 9.8 and 10.4 (p = 0.0018 and 4.9e-05), while 5.5 (p = 0.50) and 9.3 (p = 0.052) are not, so the outcome there would be partial. Without three more units whose isolation is doubtful at one step (887_127 CG1L, 897_041 CG1L, 887_005 CG2L; 7 units left), the acid ANOVA gives p = 0.15, as in the main reading.

## cg-state-transitions

Blocked (`metadata_insufficient`): the criterion counts transitions among CG states 1 to 3 (SC and LC, SC only, silent) over whole recordings. The annotation is not in the dataset, and the proxy follows the LC only, so it cannot tell state 2 from state 3.

## fig7c-cg-frequency

> Cardiac rhythm frequency declined in both acidic and basic saline and was significantly slower at pH 10.4.

Criterion: readings (a) both ANOVAs p < 0.05; (b) base ANOVA p < 0.05 and the mean at 5.5 below the acid control; reproduced when under one reading the corrected t-test at 10.4 has p < 0.05, lower.

In [ ]:
for lab, df in [("main", measures[("CG", "hz", "zero")]), ("missing steps left out", measures[("CG", "hz", "nan")]),
                ("code window", code_measures[("CG", "hz")]), ("CG burst rule 0.3 s, 2 spikes", cg03["hz"])]:
    a, b, t = C.rm_report(df, f"LC burst frequency (Hz), {lab}", show=(lab == "main"))
    ra = a["p"] < 0.05 and b["p"] < 0.05
    rb = b["p"] < 0.05 and t.loc["5.5", "diff"] < 0
    t104 = t.loc["10.4", "p_corr"] < 0.05 and t.loc["10.4", "diff"] < 0
    print(f"{lab}: acid p = {a['p']:.3g}, base p = {b['p']:.3g}; reading (a) {ra}, (b) {rb}; 10.4 diff {t.loc['10.4', 'diff']:.3f} p_corr {t.loc['10.4', 'p_corr']:.3g} -> {t104}")

**Result: not reproduced (assumed: automatic spike thresholds and the CG burst rule).** The mean LC frequency at 10.4 is 0.092 Hz below control, but the corrected t-test is not significant (p = 0.84). The base ANOVA gives p = 0.056, and neither reading's ANOVA condition holds. No variant (missing steps left out, the code's window, the 0.3 s burst rule) makes 10.4 significant (p = 0.18 to 1). Frequency at 5.5 is 0.059 Hz below the acid control. Without the four units of doubtful isolation, the base ANOVA gives p = 2.9e-09 and 10.4 is significantly slower (-0.171 Hz, corrected p = 8.7e-04), so this outcome depends on those units. Without three more (7 units), 10.4 is again significant (-0.189 Hz, p = 9.4e-04).

## fig7d-cg-spikes-per-burst

> Acid and base both significantly affected LC spikes per burst, which increased significantly at pH 5.5 and 6.1 and decreased at pH 9.8 and 10.4.

In [ ]:
for lab, df in [("main", measures[("CG", "spikes", "zero")]), ("missing steps left out", measures[("CG", "spikes", "nan")]),
                ("code window", code_measures[("CG", "spikes")]), ("CG burst rule 0.3 s, 2 spikes", cg03["spikes"])]:
    a, b, t = C.rm_report(df, f"LC spikes per burst, {lab}", show=(lab == "main"))
    tt = {"5.5 up": t.loc["5.5", "p_corr"] < 0.05 and t.loc["5.5", "diff"] > 0, "6.1 up": t.loc["6.1", "p_corr"] < 0.05 and t.loc["6.1", "diff"] > 0,
          "9.8 down": t.loc["9.8", "p_corr"] < 0.05 and t.loc["9.8", "diff"] < 0, "10.4 down": t.loc["10.4", "p_corr"] < 0.05 and t.loc["10.4", "diff"] < 0}
    print(f"{lab}: acid p = {a['p']:.3g}, base p = {b['p']:.3g}; t-tests {tt}")

**Result: not reproduced (assumed: automatic spike thresholds and the CG burst rule).** The base ANOVA is significant (p = 3.7e-05), and LC spikes per burst fall at 10.4 (-2.84, corrected p = 9.2e-04). The acid ANOVA is not (p = 0.54), so [RM] runs no acid t-tests; the increases at 5.5 (+1.27) and 6.1 (+0.61) are not significant either. The fall at 9.8 is not significant (-2.06, p = 0.26). The acid ANOVA fails under every variant (p 0.13 to 0.54), and without the four units of doubtful isolation (p = 0.20). Without three more (7 units), both ANOVAs are significant (acid p = 0.020, base p = 1.1e-06) and so are the falls at 9.8 and 10.4 (p = 0.022 and 0.019), while the rises at 5.5 (+1.65, p = 0.29) and 6.1 (+1.20, p = 0.37) are not, so the outcome there would be partial.

## fig7e-cg-duty-cycle

> Base, but not acid, had a significant effect on LC duty cycle.

In [ ]:
for lab, df in [("main", measures[("CG", "duty", "zero")]), ("missing steps left out", measures[("CG", "duty", "nan")]),
                ("code window", code_measures[("CG", "duty")]), ("CG burst rule 0.3 s, 2 spikes", cg03["duty"])]:
    a, b, t = C.rm_report(df, f"LC duty cycle, {lab}", show=(lab == "main"))
    print(f"{lab}: base p = {b['p']:.3g} (< 0.05: {b['p'] < 0.05}); acid p = {a['p']:.3g} (>= 0.05: {a['p'] >= 0.05}); "
          f"acid diffs {t.loc[C.ACID_STEPS, 'diff'].round(3).to_dict()}")

**Result: reproduced (assumed: automatic spike thresholds and the CG burst rule).** Base ANOVA p = 4.8e-05 and acid ANOVA p = 0.52 (14 units), and the outcome is the same under every variant shown (base p 5.0e-04 to 0.0091; acid p 0.13 to 0.52). The acid effects are at most 0.014 in duty cycle. Without the four units of doubtful isolation (base p = 1.7e-06, acid p = 0.33) or three more (base p = 7.8e-05, acid p = 0.50) the outcome is the same.

## fig7s2-cg-control-variability

> Cardiac frequency, LC spikes per burst and LC duty cycle vary widely between preparations at control pH, independent of pH.

In [ ]:
v = control_variability("CG")
print(v.to_string())
print("reading (a) for all three:", bool(v.reading_a.all()), "; reading (b) for all three:", bool(v.reading_b.all()))

**Result: reproduced under reading (b) (assumed, as fig7c).** The Kruskal-Wallis test of the control bins on unit gives H = 625, 516 and 468 for the three measures (14 units; p prints as 0). Under reading (a), the preparation means span 0.39, 0.79 and 0.62 of the pooled range, so (a) fails for frequency.

## fig8a-ganglion-difference

> The fraction of time with a normal rhythm peaks near control in both ganglia, but the CG is more sensitive to extreme base and the STG to extreme acid; pH, ganglion and their interaction all had significant effects in both acid and base, and the pyloric rhythm was significantly less rhythmic than the cardiac at pH 5.5 and more rhythmic at pH 9.3 and 9.8.

Criterion: [MIX] on the per-preparation fraction rhythmic (proxy) of both ganglia, readings (a) sum-to-zero and (b) treatment contrasts for ganglion, each with (c) univariate and (d) multivariate tests; Welch t-tests per pH, corrected by 4 (acid) or 5 (base).

In [ ]:
def fig8a(stg, cg, show=True):
    out = {}
    for coding in ("sum", "treatment"):
        for proto, cols in (("acid", C.ACID), ("base", C.BASE)):
            m = C.mixed_anova(cg, stg, cols, coding)
            for eff in ("pH", "ganglion", "interaction"):
                r = m[eff]
                out[(coding, proto, eff)] = dict(F_uni=r["F"], p_uni=r["p"], pillai=r.get("pillai", np.nan), p_multi=r.get("p_pillai", r["p"]), n=m["n"])
    tab = pd.DataFrame(out).T
    welch = {}
    for s, k in (("5.5", 4), ("9.3", 5), ("9.8", 5), ("6.1", 4), ("6.7", 4), ("7.2", 4), ("8.3", 5), ("8.8", 5), ("10.4", 5)):
        tt = stats.ttest_ind(stg[s], cg[s], equal_var=False, nan_policy="omit")
        welch[s] = dict(stg=stg[s].mean(), cg=cg[s].mean(), t=tt.statistic, p_corr=min(1.0, tt.pvalue * k))
    welch = pd.DataFrame(welch).T
    if show:
        print(tab.assign(F_uni=tab.F_uni.astype(float).round(2), p_uni=tab.p_uni.map("{:.3g}".format), pillai=tab.pillai.astype(float).round(3),
                         p_multi=tab.p_multi.map("{:.3g}".format)).to_string())
        print(welch.round(4).to_string())
    return tab, welch
stg_r, cg_r = rhythm[("STG", "zero")], rhythm[("CG", "zero")]
tab, welch = fig8a(stg_r, cg_r)
tt_ok = welch.loc["5.5", "stg"] < welch.loc["5.5", "cg"] and welch.loc["5.5", "p_corr"] < 0.05
tt_ok = [tt_ok] + [welch.loc[s, "stg"] > welch.loc[s, "cg"] and welch.loc[s, "p_corr"] < 0.05 for s in ("9.3", "9.8")]
for coding in ("sum", "treatment"):
    for col in ("p_uni", "p_multi"):
        sig = [tab.loc[(coding, p, e), col] < 0.05 for p in ("acid", "base") for e in ("pH", "ganglion", "interaction")]
        print(f"{coding} contrasts, {col}: effects p < 0.05 = {sum(sig)} of 6")
print("t-tests (STG lower at 5.5, higher at 9.3 and 9.8):", tt_ok)
print("mean fraction rhythmic, STG:", stg_r.mean().round(3).to_dict()); print("CG:", cg_r.mean().round(3).to_dict())
for f in (2.0, 5.0):
    print(f"variant proxy factor {f}:"); fig8a(proxy_var[f]["STG"], proxy_var[f]["CG"])
print("variant: missing steps left out (a preparation without a step is dropped from that protocol's ANOVA and from that step's t-test)")
fig8a(rhythm[("STG", "nan")], rhythm[("CG", "nan")])
print("variant: STG by PD bursting alone")
_ = fig8a(stg_pd["zero"], cg_r)
for lab, kept in (("CG without the four units of doubtful isolation", keep), ("CG without those and three more", keep7)):
    print("variant:", lab)
    _ = fig8a(stg_r, cg_r.loc[kept])

**Result: not reproduced, on the state readings (assumed).** With sum-to-zero contrasts, all six effects are significant under both univariate and multivariate tests: pH (p = 2.0e-10 acid, 2.8e-12 base), ganglion (0.027, 0.015) and interaction (2.7e-04, 0.0040). With treatment contrasts, the acid pH effect is not (p = 0.15). The STG is less rhythmic than the CG at 5.5 (0.37 against 0.82, corrected p = 0.029). It is more rhythmic at 9.3 (0.85 against 0.55) and 9.8 (0.77 against 0.40), but not significantly (p = 0.36 and 0.21). The criterion needs all three t-tests, and its partial condition also needs them. With gap factor 2, 9.8 becomes significant (p = 0.023) but 5.5 does not (p = 0.070); with factor 5, 5.5 holds (p = 0.009) and 9.3 and 9.8 do not. With missing steps left out, the ganglion effects are not significant (p = 0.11 and 0.12) and 5.5 is not either (p = 0.12). With PD bursting alone the result is as in the main reading (5.5 p = 0.042). The ANOVA part also depends on the gap factor: with sum contrasts, the acid ganglion effect gives p = 0.14 at factor 2 and the base ganglion effect p = 0.061 at factor 5. Without the four CG units of doubtful isolation, 5.5 (p = 0.053), 9.3 (0.54) and 9.8 (0.064) all fail; without three more, 5.5 holds (p = 0.013) and 9.3 and 9.8 fail (0.47 and 0.25). Every variant fails the three t-tests the criterion needs.

## fig8b-individual-preparations

> All preparations were rhythmic at control; 14 of 15 STG preparations stopped firing after 15 min at pH 5.5, only 6 of 15 CG preparations lost rhythmic activity at pH 5.5, and 3 of 15 CG preparations kept a normal rhythm in every pH condition.

Expected counts 14 (± 1.4), 6 (± 0.6), 3 (± 0.3). Readings as in the criterion. "Ceased firing" (a) state 4 at the end of the step is read as no PD spikes in the last 10 s; (b) no PD bursts in the last minute; (c) fraction rhythmic 0. The STG preparations lacking 5.5 (877_093, 887_069) count as not firing under the code's zero rule and are shown apart.

In [ ]:
rows = []
for u in units:
    if u["ganglion"] != "STG" or u["out"] is None:
        continue
    v = u["out"]["5.5"]
    if v is None:
        rows.append(dict(prep=u["label"], has_55=False)); continue
    r = R[v["file"]]["ext"][u["channel"]]
    spikes_last10 = int(((r["late"][0] >= v["seconds"] - 10) & (r["late"][1] >= v["thr"])).sum())
    bursts_lastmin = int((v["bursts"].tstart >= v["seconds"] - 60).sum())
    rows.append(dict(prep=u["label"], has_55=True, spikes_last10s=spikes_last10, bursts_last_min=bursts_lastmin, frac_rhythmic=v["rhythmic"].mean(),
                     frac_pd_alone=v["rhythmic_pd"].mean()))
s55 = pd.DataFrame(rows)
print(s55.round(3).to_string(index=False))
have = s55[s55.has_55]
cease = {"a": int((have.spikes_last10s == 0).sum()), "b": int((have.bursts_last_min == 0).sum()), "c": int((have.frac_rhythmic == 0).sum())}
n_missing = int((~s55.has_55).sum())
print("STG ceased at 5.5, of", len(have), "with a 5.5 file:", cease, f"; plus {n_missing} without a 5.5 file (counted as 0 by the code's rule)")
cg = rhythm[("CG", "zero")]
red = {"a": int((cg["5.5"] < 1).sum()), "b": int((cg["5.5"] < cg["ctl_acid"]).sum())}
normal = {"a": int((cg[C.COLUMNS] >= 1).all(axis=1).sum()), "b": int((cg[["5.5", "ctl_acid", "10.4"]] >= 1).all(axis=1).sum())}
ctl = {"STG a": int((stg_r["ctl_acid"] >= 1).sum()), "STG b": int((stg_r["ctl_acid"] >= 0.5).sum()), "CG a": int((cg["ctl_acid"] >= 1).sum()), "CG b": int((cg["ctl_acid"] >= 0.5).sum())}
print("CG with reduced rhythm at 5.5 (of", len(cg), "):", red, "; CG normal in every condition:", normal)
print("rhythmic at control before acid (counts):", ctl, "of", len(stg_r), "STG and", len(cg), "CG")
print("Figure 8B STG control column is the recovery from acid in pHCumulative.m:", int((stg_r["rec_acid"] >= 1).sum()), "of", len(stg_r), "STG fully rhythmic there")
for f in (2.0, 5.0):
    cgv, stv = proxy_var[f]["CG"], proxy_var[f]["STG"]
    print(f"proxy factor {f}: CG reduced at 5.5 (a) {int((cgv['5.5'] < 1).sum())}, (b) {int((cgv['5.5'] < cgv['ctl_acid']).sum())}; "
          f"CG normal everywhere (a) {int((cgv[C.COLUMNS] >= 1).all(axis=1).sum())}, (b) {int((cgv[['5.5', 'ctl_acid', '10.4']] >= 1).all(axis=1).sum())}; "
          f"STG fraction 0 at 5.5 {int((stv.loc[have.prep, '5.5'] == 0).sum())}")
print("STG fraction 0 at 5.5 by PD bursting alone (reading c):", int((have.frac_pd_alone == 0).sum()), "of", len(have))
cgn = rhythm[("CG", "nan")]
print("variant, missing steps left out: CG reduced at 5.5 (a)", int((cgn["5.5"] < 1).sum()), "(b)", int((cgn["5.5"] < cgn["ctl_acid"]).sum()),
      "of", int(cgn["5.5"].notna().sum()), "; CG normal in every condition it has (a)", int((cgn[C.COLUMNS].fillna(1) >= 1).all(axis=1).sum()),
      "(b)", int((cgn[["5.5", "ctl_acid", "10.4"]].fillna(1) >= 1).all(axis=1).sum()), "; units missing a step:", list(cgn.index[cgn[C.COLUMNS].isna().any(axis=1)]))

**Result: not reproduced, on the state readings (assumed).** Of 11 STG preparations with a 5.5 file, 6 have no PD spikes in its last 10 s (reading a) and 6 no PD bursts in its last minute (b). 4 have a rhythmic fraction of 0 (c), as with PD bursting alone. Counting the 2 preparations that lack 5.5 as silent gives at most 8 of 13, against 14 of 15 (tolerance 1.4). 887_141 and 897_037 are rhythmic throughout 5.5 (fraction 1.0) and 877_141 for most of it (0.954). Of 14 CG units, 5 have a reduced fraction at 5.5 (expected 6 ± 0.6; 3 under gap factor 5), and 2 are fully rhythmic in every condition (expected 3 ± 0.3; 3 under gap factor 5, reading a). With missing steps left out, CG reduced at 5.5 is 5 and normal everywhere 2 under (a), 4 under (b). Every preparation is fully rhythmic at control.

## fig8b-cross-predictive

> STG preparations with reduced activity in base were very susceptible to loss of activity in extreme acid, while in the CG it was the reverse, activity in acid predicting activity in base.

Criterion reading (a): every STG preparation with fraction below 1 at 10.4 has fraction 0 at 5.5, every CG preparation below 1 at 5.5 is below 1 at 10.4, and the reverse implication fails in each ganglion for at least one preparation; (b) Fisher tables shown.

In [ ]:
def cross(fr, g):
    red55, red104 = fr["5.5"] < 1, fr["10.4"] < 1
    tab = pd.crosstab(red55.rename("reduced at 5.5"), red104.rename("reduced at 10.4"))
    print(g); print(tab.to_string())
    full = np.array([[((~red55) & (~red104)).sum(), ((~red55) & red104).sum()], [(red55 & ~red104).sum(), (red55 & red104).sum()]])
    print("Fisher exact p:", stats.fisher_exact(full).pvalue)
    return red55, red104
r55, r104 = cross(stg_r, "STG")
stg_imp = bool((stg_r.loc[r104, "5.5"] == 0).all()); stg_rev_fails = bool(((stg_r["5.5"] == 0) & ~r104).any())
print("STG: every one reduced at 10.4 has 0 at 5.5:", stg_imp, list(stg_r.index[r104]), "; reverse fails for one:", stg_rev_fails)
c55, c104 = cross(cg_r, "CG")
cg_imp = bool(c104[c55].all()); cg_rev_fails = bool((c104 & ~c55).any())
print("CG: every one reduced at 5.5 is reduced at 10.4:", cg_imp, "; reverse fails for one:", cg_rev_fails)
print("reading (a) holds:", stg_imp and stg_rev_fails and cg_imp and cg_rev_fails)
for lab, s_fr, c_fr in (("missing steps left out", rhythm[("STG", "nan")].dropna(subset=["5.5", "10.4"]), rhythm[("CG", "nan")].dropna(subset=["5.5", "10.4"])),
                        ("STG by PD bursting alone", stg_pd["zero"], cg_r)):
    print("variant:", lab)
    a55, a104 = cross(s_fr, "STG"); b55, b104 = cross(c_fr, "CG")
    print("  STG implication:", bool((s_fr.loc[a104, "5.5"] == 0).all()), "; reverse fails:", bool(((s_fr["5.5"] == 0) & ~a104).any()),
          "; CG implication:", bool(b104[b55].all()), "; reverse fails:", bool((b104 & ~b55).any()))

**Result: not reproduced, on the state readings (assumed).** Of the 4 STG preparations reduced at 10.4, 877_093, 877_121, 887_005 and 887_137, not all have a fraction of 0 at 5.5. In the CG, 1 of 5 units reduced at 5.5 is not reduced at 10.4. Neither Fisher test is significant (p = 1.0 in both ganglia). Reading (a) fails in both ganglia with missing steps left out and with PD bursting alone too.

## baseline-frequency-correlations

> Frequency at control correlates strongly with frequency in extreme acid and in extreme base for PD neurons, and LC frequency at control correlates strongly with LC frequency in extreme acid but not in extreme base.

Readings: PD data (a) extracellular STG, (b) PTX PD burst frequency; correlation Pearson or Spearman; "strong" (a) positive with p < 0.05, (b) r ≥ 0.7. Control just before each protocol against 5.5 and 10.4. A missing step is left out of its correlation.

In [ ]:
def corr(df, ctl, ext):
    d = df[[ctl, ext]].dropna()
    return dict(n=len(d), pearson=stats.pearsonr(d[ctl], d[ext]), spearman=stats.spearmanr(d[ctl], d[ext]))
sets = {"PD (a) extracellular": measures[("STG", "hz", "nan")], "PD (b) PTX": ptx[("PD", "burst_hz_trough")], "LC": measures[("CG", "hz", "nan")]}
rows = []
for name, df in sets.items():
    for proto, ctl, ext in (("acid", "ctl_acid", "5.5"), ("base", "ctl_base", "10.4")):
        c = corr(df, ctl, ext)
        for kind in ("pearson", "spearman"):
            rr = c[kind]
            rows.append(dict(data=name, protocol=proto, method=kind, n=c["n"], r=rr.statistic, p=rr.pvalue))
cr = pd.DataFrame(rows)
print(cr.assign(r=cr.r.round(3), p=cr.p.map("{:.3g}".format)).to_string(index=False))
def strong(r, p, rule): return (r > 0 and p < 0.05) if rule == "a" else r >= 0.7
combos = []
for pdset in ("PD (a) extracellular", "PD (b) PTX"):
    for kind in ("pearson", "spearman"):
        for rule in ("a", "b"):
            g = lambda d, pr: cr[(cr.data == d) & (cr.protocol == pr) & (cr.method == kind)].iloc[0]
            ok = (strong(g(pdset, "acid").r, g(pdset, "acid").p, rule) and strong(g(pdset, "base").r, g(pdset, "base").p, rule)
                  and strong(g("LC", "acid").r, g("LC", "acid").p, rule) and not strong(g("LC", "base").r, g("LC", "base").p, rule))
            combos.append(dict(pd_data=pdset, method=kind, strong=rule, holds=ok))
print(pd.DataFrame(combos).to_string(index=False))

**Result: not reproduced (assumed: automatic spike thresholds).** LC frequency at control correlates with LC frequency at 5.5 (Pearson r = 0.73, p = 0.0033; 14 units) and not at 10.4 (r = -0.16, p = 0.66), as claimed. For PD, the extracellular frequencies correlate weakly and not significantly (r = 0.40 at 5.5, 0.48 at 10.4; p 0.22 and 0.12). In the PTX preparations the correlation at 5.5 is negative (r = -0.81) and at 10.4 r = 0.34. No combination of readings makes both PD correlations strong.

## robust-6-1-to-8-8

> Both the pyloric and cardiac rhythms remained robust from about pH 6.1 to pH 8.8.

Criterion: a claim of absence on the paper's tests: in [RM] of fraction rhythmic (proxy) for each ganglion, no corrected paired t-test at 6.1, 6.7, 7.2, 8.3 or 8.8 shows a significant decrease.

In [ ]:
for g in ("STG", "CG"):
    a, b, t = C.rm_report(rhythm[(g, "zero")], f"{g} fraction rhythmic", show=False)
    sub = t.loc[["6.1", "6.7", "7.2", "8.3", "8.8"]]
    print(g, "means:", sub.mean_step.round(3).to_dict(), "; corrected p:", sub.p_corr.round(3).to_dict(),
          "; significant decreases:", int(((sub.p_corr < 0.05) & (sub["diff"] < 0)).sum()))
    for f in (2.0, 5.0):
        _, _, tv = C.rm_report(proxy_var[f][g], "", show=False)
        print(f"   proxy factor {f}: significant decreases 6.1 to 8.8:", int(((tv.loc[['6.1', '6.7', '7.2', '8.3', '8.8'], 'p_corr'] < 0.05) & (tv.loc[['6.1', '6.7', '7.2', '8.3', '8.8'], 'diff'] < 0)).sum()))
for lab, df in (("STG by PD bursting alone", stg_pd["zero"]), ("STG, 1 lvn spike per cycle", stg_other[1]), ("STG, 4 lvn spikes per cycle", stg_other[4])):
    _, _, tv = C.rm_report(df, "", show=False)
    sub = tv.loc[["6.1", "6.7", "7.2", "8.3", "8.8"]]
    print(f"   {lab}: significant decreases 6.1 to 8.8:", int(((sub.p_corr < 0.05) & (sub["diff"] < 0)).sum()), "; smallest corrected p", round(sub.p_corr.min(), 3))

**Result: reproduced, on the state readings (assumed).** No corrected t-test from 6.1 to 8.8 shows a significant decrease in either ganglion (STG smallest corrected p 0.53, CG 0.68), under gap factors 2, 3 and 5, with 1 or 4 lvn spikes per cycle and with PD bursting alone. The mean fractions there are 0.86 to 1.0 (STG) and 0.91 to 1.0 (CG). The CG acid ANOVA is not significant (p = 0.058), so [RM] would run no acid t-tests there; this does not change the count of decreases.

## intact-depolarize-in-acid

> In intact (no PTX) preparations both PD and LP neurons depolarized in acid, unlike the semi-isolated neurons where only PD did.

Criterion: intact STG preparations with intracellular PD and LP, minimum membrane potential of [INTRA] in the last minute, [RM] on the acid protocol; readings (a) acid ANOVA p < 0.05 and higher mean at 5.5 than control, (b) higher at 5.5 than control in most preparations; for both neurons. Preparations lacking 5.5 are left out of the ANOVA.

In [ ]:
for unit in ("PD", "LP"):
    for k in ("vmin_methods", "vmin_code"):
        df = intact[(unit, k)]
        a, b, t = C.rm_report(df, f"intact {unit} minimum Vm ({k})", show=(k == "vmin_methods"))
        d = (df["5.5"] - df["ctl_acid"]).dropna()
        print(f"{unit} {k}: preparations {len(df)} ({list(df.index)}); with 5.5: {len(d)}; acid ANOVA p = {a['p']:.3g}; "
              f"mean 5.5 minus control {d.mean():.2f} mV; higher at 5.5 in {int((d > 0).sum())} of {len(d)}; "
              f"(a) {a['p'] < 0.05 and d.mean() > 0}; (b) {(d > 0).sum() > len(d) / 2}")

**Result: reproduced (assumed: the code's starting parameters for the waveform analysis).** In intact preparations, PD (7 with 5.5 of 9) and LP (5 of 6) are depolarized at 5.5 against the acid control, by 18.2 and 11.7 mV. Every preparation is depolarized, and the acid ANOVAs give p = 2.9e-11 (PD) and 6.9e-05 (LP). The code's slow-wave troughs give the same outcome.

In [ ]:
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2a-stg1-traces | reproduced (assumed) |
| fig2bcd-stg1-measures | reproduced (assumed) |
| fig3a-stg-variability-examples | blocked, metadata_insufficient |
| fig3c-stg-rhythmic-fraction | not reproduced (assumed) |
| stg-state-transitions | blocked, metadata_insufficient |
| fig3d-stg-frequency | not reproduced (assumed) |
| fig3e-stg-spikes-per-burst | reproduced (assumed) |
| fig3f-stg-duty-cycle | not reproduced (assumed) |
| fig3s2-stg-control-variability | reproduced (assumed) |
| fig4a-ptx-example | reproduced (assumed) |
| fig4bc-membrane-potential | reproduced (assumed) |
| fig4de-spike-amplitude | partially reproduced (assumed) |
| fig4fg-ptx-frequency | partially reproduced (assumed) |
| gastric-like-rhythm-counts | blocked, methods_underspecified |
| gastric-conditions-co-occur | blocked, methods_underspecified |
| fig5-gastric-recovery-example | blocked, methods_underspecified |
| fig6-cg1-measures | not reproduced (assumed) |
| fig6a-cg2-less-rhythmic | reproduced (assumed) |
| fig7b-cg-rhythmic-fraction | not reproduced (assumed) |
| cg-state-transitions | blocked, metadata_insufficient |
| fig7c-cg-frequency | not reproduced (assumed) |
| fig7d-cg-spikes-per-burst | not reproduced (assumed) |
| fig7e-cg-duty-cycle | reproduced (assumed) |
| fig7s2-cg-control-variability | reproduced (assumed) |
| fig8a-ganglion-difference | not reproduced (assumed) |
| fig8b-individual-preparations | not reproduced (assumed) |
| fig8b-cross-predictive | not reproduced (assumed) |
| baseline-frequency-correlations | not reproduced (assumed) |
| robust-6-1-to-8-8 | reproduced (assumed) |
| intact-depolarize-in-acid | reproduced (assumed) |

How the analysis differs from the paper's:

- Spikes are found with automatic thresholds (an Otsu split for PD on the pdn; a fraction of the largest events for the LC on the CG trunk), where the authors set thresholds by hand in Spike2. CG bursts use a 1.0 s maximum interval and one spike, where the code's starting values are 0.3 s and two spikes (shown as a variant).
- The manual state annotations are not in the dataset. States are read from the spikes: for the STG, PD bursting on the pdn with LP and PY firing on the lvn between the PD bursts (states 1 and 2 together, without telling them apart); for the CG, LC bursting (state 1), which cannot separate the CG states 2 and 3.
- The LC isolation is automatic and is doubtful in four units (877_149 CG1L and CG2L, 887_149 CG2L, 897_041 CG3L; height histograms above). Without them fig7c would hold at 10.4 and fig7b would be partial; without three more units whose isolation is doubtful at one step (887_127 CG1L, 897_041 CG1L, 887_005 CG2L), fig7c would hold and fig7d would be partial. These CG outcomes depend on the isolation.
- 13 STG preparations (877_127 is not in the project; 877_101 has no pdn) and 14 CG units (887_017 is not in the project) in place of 15 each. A missing pH step counts as 0 in the extracellular measures, as the paper's code gives it; leaving it out is shown as a variant.
- The PTX files have no README; their pH is taken from the recorded bath pH, and a PTX preparation missing a step is left out of that protocol's tests.
- The [INTRA] parameters are the code's starting values, which the authors adjusted by hand.

What was found about the dataset: two experiments the paper's code uses are absent (877_127, 887_017). Seven experiments lack one pH step. The PTX folders have no README, and in two of them the recorded pH shows the 9.3 file before the 8.8 file. In 903_046 one file names the PD and LP electrodes the other way round from the other ten, and nothing in the file shows it. In 903_052 the bath pH electrode responds less than in the other PTX experiments. Several READMEs disagree with their folders (887_145 lists 887_149 file names; 877_149 lists 877_149_00010.abf; two folders hold unlisted files; 887_061 names an IN 15 channel the files do not have). 877_101 has no pdn. The state, burst and gastric annotations behind the paper's figures are not deposited. Every STG and CG file read whole matches its pinned SHA-256.